# 📘 Notebook 18: Measuring Scale

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module E: A Real Warehouse, End to End · Notebook 6 of 6 in this module · (18 of 18 overall)*

---

> 🧭 **How to use this notebook.** Run the cells in order, from top to bottom: click a cell and press **Shift+Enter**. After most cells there is a short paragraph headed **What you should see**: compare it with your own output before you move on. If your numbers differ, stop and find out why, because that is how data work is learned. The exercises at the end start easy and get harder: first **warm-up** questions, then **think it through** questions (what would you run, and why?), then **technical** tasks. Try each one before you open its solution.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Estimate how much memory a table needs, and how many rows one machine can hold
- Run one analytical question in **pandas**, **SQLite** and **Spark** on growing copies of the same data, and time each fairly
- Separate a tool's **fixed overhead** from its **cost per row**, and find where the lines cross
- Explain why a cluster engine loses on small data, and what it offers when the data grows
- Turn the measurements into a reasoned recommendation, the kind a platform evaluation needs

> **Prerequisites:** Notebooks 14 to 16 (this warehouse in SQLite, pandas and Spark). Notebooks 07, 09, 10 and 12 discuss memory limits, Spark and the cost of the cloud in more depth.
>
> 🔭 **Why this notebook matters:** every claim about *performance at scale* deserves a measurement, and the heavy machinery is not always faster. Here we settle it for the AdventureWorks warehouse itself. We make the sales table 1, 8, 64 and 256 times larger, ask the same question of three tools, and plot the results. The plot is small, but the way of thinking behind it is how platform decisions should be made, and it is the evidence a report on *performance at scale* or *cost efficiency* ought to contain.

> 📦 **Libraries used in this notebook.** **pandas**, **SQLite**, **PySpark** (local mode), **PyArrow** for Parquet files, **psutil** to read the machine's memory, and **Matplotlib**. The cell below installs what is missing; in Google Colab everything is already present.
>
> ⏱️ The experiment of Section 2 takes several minutes, most of it at the largest size. That is part of the lesson.

> 🗃️ **The data.** This module works with **AdventureWorks**, the sample data warehouse that Microsoft publishes for learning SQL Server: the online sales of an imaginary bicycle company, with its customers, products and dates. It is open data (MIT licence) and much closer to a real warehouse than anything we could invent. The cell below downloads it from Microsoft's GitHub repository (about 17 MB, once per session) and saves each table as an ordinary CSV file.

In [ ]:
%pip install -q pyspark pyarrow psutil

In [ ]:
import io
import os
import re
import urllib.request
import zipfile

import pandas as pd

AW_URL = ("https://github.com/microsoft/sql-server-samples/releases/download/"
          "adventureworks/AdventureWorksDW-data-warehouse-install-script.zip")
AW_TABLES = ["FactInternetSales", "DimCustomer", "DimGeography", "DimSalesTerritory", "DimProduct",
             "DimProductSubcategory", "DimProductCategory", "DimDate", "DimPromotion", "DimCurrency"]


def fetch_adventureworks(folder="adventureworks_dw"):
    """Download Microsoft's AdventureWorks sample warehouse once, and save each table as a CSV file with a header."""
    if os.path.isdir(folder):
        return folder
    archive = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(AW_URL).read()))
    script = archive.read("instawdbdw.sql")
    script = script.decode("utf-16" if script[:2] in (b"\xff\xfe", b"\xfe\xff") else "utf-8-sig")
    os.makedirs(folder)
    for table in AW_TABLES:
        # The column names are not in the data files: they are in the CREATE TABLE statements of the install script.
        body = re.search(r"CREATE TABLE \[dbo\]\.\[" + table + r"\]\((.*?)\n\)", script, re.S).group(1)
        columns = re.findall(r"^\s*\[(\w+)\]", body, re.M)
        text = archive.read(table + ".csv").decode("utf-8-sig")
        df = pd.read_csv(io.StringIO(text), sep="|", header=None, names=columns, dtype=str, keep_default_na=False)
        df = df.drop(columns=[c for c in ("LargePhoto", "SalesTerritoryImage") if c in df])   # pictures as hex text
        df.replace("", None).to_csv(f"{folder}/{table}.csv", index=False)
    return folder


def load(name, folder=None):
    """Read one table into pandas. Other exports of this warehouse are sometimes saved as Latin-1, so try both."""
    path = f"{folder or DATA}/{name}.csv"
    try:
        return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", low_memory=False)


DATA = fetch_adventureworks()
print("Tables:", sorted(f[:-4] for f in os.listdir(DATA)))

In [ ]:
import gc
import sqlite3
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import psutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
print(f"This machine: {psutil.cpu_count()} cores, {psutil.virtual_memory().total / 1e9:.1f} GB of memory")

**What you should see:** the ten table names, then the size of the machine. A free Colab session has 2 cores and about 13 GB of memory; a laptop typically has 4 to 16 cores and 8 to 32 GB. Keep your numbers: every result below depends on them.

## 1. How big is the data, really?

### Intuition first
Before choosing a tool, measure the bytes. A table in memory costs a certain number of **bytes per row**. Divide the machine's memory by it, keep a safety margin, and you know roughly how many rows one machine can work with.

`memory_usage(deep=True)` asks pandas how many bytes each column really uses, including the text. We measure the full sales table, then a **slim** copy with only the four columns our question needs, stored in compact types: `int16` holds whole numbers up to 32,767 in 2 bytes, `int32` up to about 2 billion in 4 bytes, `float64` a decimal number in 8 bytes.

In [ ]:
sales = load("FactInternetSales")
per_row = sales.memory_usage(deep=True).sum() / len(sales)
print(f"FactInternetSales: {len(sales)} rows, {per_row:.0f} bytes per row in pandas, all 26 columns")

slim = pd.DataFrame({
    "year": pd.to_datetime(sales["OrderDate"]).dt.year.astype("int16"),
    "product_key": sales["ProductKey"].astype("int32"),
    "customer_key": sales["CustomerKey"].astype("int32"),
    "revenue": sales["SalesAmount"].astype("float64"),
})
slim_row = slim.memory_usage(deep=True).sum() / len(slim)
print(f"Only the four columns we need, with compact types: {slim_row:.0f} bytes per row")

usable = psutil.virtual_memory().total * 0.5          # leave half for the system and for intermediate results
for label, size in [("all columns", per_row), ("four compact columns", slim_row)]:
    print(f"{label:<22} about {usable / size / 1e6:,.0f} million rows fit in half of this machine's memory")

**What you should see:** about 480 bytes per row for the full table (text columns such as order numbers and dates are expensive), but only **18** bytes per row for the four compact columns. On a machine with 16 GB, about 18 million full rows fit in half the memory, but about 470 million slim rows. (A free Colab session has a little less memory, so its numbers are a little smaller.)

Choosing columns and types is worth more than any engine: the slim table needs less than a twentieth of the memory per row. At the size AdventureWorks has today, its sales would have to grow by a factor of thousands before one ordinary machine struggled. Keep that factor in mind while the experiment runs.

## 2. The experiment

### Intuition first
One question, typical of a warehouse: **revenue and number of distinct customers by year and product category**. It needs a join (sales to category) and a grouping with a distinct count, which is the expensive kind of aggregation, because every customer seen must be remembered.

We make copies of the sales: *k* copies, each with its own customer numbers, so that the distinct counts grow too. Then each tool answers the same question on the same rows, and we time **only the question**, not the loading, which each tool does differently.

A fair race has rules, and here they are:

* **same question, same rows** for every tool, and a check that all three give the **same answer**;
* **best of three** runs (`best_of`), because a single time can be disturbed by anything else the machine does;
* Spark reads the data from a **Parquet** file, as it would on a cluster, and runs each query once **before** timing it, so that its start-up costs are not counted against it.

First the data and the timer:

In [ ]:
category = (load("DimProduct")[["ProductKey", "ProductSubcategoryKey"]]
            .merge(load("DimProductSubcategory")[["ProductSubcategoryKey", "ProductCategoryKey"]], on="ProductSubcategoryKey")
            .merge(load("DimProductCategory")[["ProductCategoryKey", "EnglishProductCategoryName"]], on="ProductCategoryKey")
            [["ProductKey", "EnglishProductCategoryName"]]
            .rename(columns={"ProductKey": "product_key", "EnglishProductCategoryName": "category"}))

def copies(k):
    """k copies of the slim sales table; each copy gets its own customers."""
    parts = [slim.assign(customer_key=slim["customer_key"] + i * 100_000) for i in range(k)]
    return pd.concat(parts, ignore_index=True)

def best_of(function, repeats=3):
    times = []
    for i in range(repeats):
        started = time.perf_counter()
        result = function()
        times.append(time.perf_counter() - started)
    return result, min(times)

print(len(category), "products with a category;", f"{len(copies(8)):,}", "rows in 8 copies")

**What you should see:** `397 products with a category; 483,184 rows in 8 copies` (8 times 60,398).

The same question in the three languages: pandas methods, SQL for SQLite, and the Spark DataFrame API. `F.broadcast` tells Spark that the category table is small and should be copied to every worker (Notebook 16).

In [ ]:
def ask_pandas(df):
    return (df.merge(category, on="product_key")
              .groupby(["year", "category"])
              .agg(revenue=("revenue", "sum"), customers=("customer_key", "nunique")))

QUESTION_SQL = '''
    SELECT s.year, c.category, SUM(s.revenue) AS revenue, COUNT(DISTINCT s.customer_key) AS customers
    FROM sales AS s JOIN category AS c ON c.product_key = s.product_key
    GROUP BY s.year, c.category'''

def ask_spark(sdf, scat):
    return (sdf.join(F.broadcast(scat), "product_key")
               .groupBy("year", "category")
               .agg(F.sum("revenue").alias("revenue"), F.countDistinct("customer_key").alias("customers"))
               .collect())

ask_pandas(slim)

**What you should see:** a table of ten rows, one per year and category, with revenue and distinct customers: for example 2013 Bikes with about 15.4 million and 2013 Accessories with about 0.67 million. This is the answer all three tools must agree on at one copy.

Now the race. For each size we build the copies, time pandas, load them into an in-memory SQLite database and time SQLite (once only: it is slow at the largest size), write them to Parquet and time Spark, and check that the three total revenues agree. **This cell takes several minutes.**

In [ ]:
spark = (SparkSession.builder.master("local[*]").appName("measuring-scale")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.driver.memory", "4g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
scat = spark.createDataFrame(category)

results = []
for k in [1, 8, 64, 256]:
    df = copies(k)

    answer_pd, t_pandas = best_of(lambda: ask_pandas(df))

    db = sqlite3.connect(":memory:")
    df.to_sql("sales", db, index=False)
    category.to_sql("category", db, index=False)
    answer_sql, t_sqlite = best_of(lambda: db.execute(QUESTION_SQL).fetchall(), repeats=1)
    db.close()

    df.to_parquet("sales_copies.parquet", index=False)
    sdf = spark.read.parquet("sales_copies.parquet")
    ask_spark(sdf, scat)                                      # warm-up run, not timed
    answer_spark, t_spark = best_of(lambda: ask_spark(sdf, scat))

    totals = [answer_pd["revenue"].sum(), sum(r["revenue"] for r in answer_spark), sum(r[2] for r in answer_sql)]
    same = bool(np.allclose(totals, totals[0], rtol=1e-9))     # sums of floats differ in the last digits
    results.append({"copies": k, "rows": len(df), "pandas_s": t_pandas, "sqlite_s": t_sqlite,
                    "spark_s": t_spark, "same_answer": same})
    print(f"{k:>4} copies, {len(df):>11,} rows:  pandas {t_pandas:6.2f} s   SQLite {t_sqlite:6.2f} s   "
          f"Spark {t_spark:6.2f} s   same answer: {same}")
    del df, sdf
    gc.collect()

results = pd.DataFrame(results)

**What you should see:** four lines, from 60,398 to 15,461,888 rows. On our 8-core laptop:

| rows | pandas | SQLite | Spark |
|---|---|---|---|
| 60,398 | 0.04 s | 0.30 s | 6.6 s |
| 483,184 | 0.23 s | 1.8 s | 7.2 s |
| 3,865,472 | 1.9 s | 14.5 s | 8.6 s |
| 15,461,888 | 10.9 s | 68 s | 14.6 s |

pandas is the fastest at every size. SQLite grows almost exactly in proportion to the rows. Spark starts slowest of all, at more than 6 seconds for 60 thousand rows, but barely grows: by the largest size it has overtaken SQLite and come within about 4 seconds of pandas. In Colab, with 2 cores, Spark will be slower than this.

Your times will differ from ours, but the order of the tools and the shape of the growth should not. `same answer: True` on every line matters as much as the times: a fast wrong answer is worthless.

A **log-log plot** (both axes logarithmic) shows all four sizes clearly, from 60 thousand to 15 million rows:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for column, label in [("pandas_s", "pandas"), ("sqlite_s", "SQLite"), ("spark_s", "Spark (local)")]:
    ax.plot(results["rows"], results[column], marker="o", label=label)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("rows in the sales table")
ax.set_ylabel("seconds for the question (log scale)")
ax.set_title("One question, three tools, growing data")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.show()

**What you should see:** three lines. pandas and SQLite climb as straight, parallel lines (their time grows with the rows), SQLite always about seven times higher. Spark's line starts high and stays almost flat, then bends upward at the largest sizes: it crosses SQLite between 1 and 4 million rows and approaches pandas at the right edge.

## 3. Reading the plot

### Intuition first
Think of a taxi fare: a fixed charge when you get in, plus a price per kilometre. Every tool's time works the same way: roughly **a fixed overhead plus a cost per row**. On a log-log plot, a tool whose time is all per-row cost climbs in a straight line; a tool dominated by overhead starts high and stays flat for a while.

`np.polyfit(x, y, 1)` fits a straight line *y = a·x + b* through the measurements: *a* is the cost per row, *b* the fixed overhead.

In [ ]:
for column in ["pandas_s", "sqlite_s", "spark_s"]:
    per_row_cost, overhead = np.polyfit(results["rows"], results[column], 1)
    print(f"{column[:-2]:<7} overhead about {max(overhead, 0):5.2f} s,  about {per_row_cost * 1e6:6.2f} s per million rows")

**What you should see:** pandas with almost no overhead and about 0.7 s per million rows; SQLite with almost no overhead and about 4.4 s per million rows; Spark with about **6.7 s of overhead** and only about **0.5 s per million rows**.

Now the plot is explained. Spark pays a fixed fare of almost 7 seconds per question, then works more cheaply per row than either other tool. If the straight lines held, Spark would catch pandas at about 6.7 / (0.71 minus 0.51), roughly 34 million rows, but that is beyond what we measured, and extrapolating a fit is a guess, not a measurement.

The reasons behind the numbers:

- **pandas** has almost no overhead and works in memory with fast, vectorised code. On data that fits in memory, it is very hard to beat.
- **SQLite** is a single-threaded row store: it reads every column of every row and sorts for the distinct count. Its cost per row is the highest, so it falls behind as the data grows. Its strengths (transactions, indexes, a file on disk) are not what this question needs.
- **Spark** pays a fixed price on every query: planning, scheduling tasks, moving data between stages. On small data that price is the whole cost. But its cost per row is spread across all the cores, and it reads only the columns it needs from Parquet, so its line rises slowly.

Whether Spark actually overtakes pandas on one machine depends on the machine and the question; on many laptops it never does while the data still fits in memory.

## 4. Where one machine stops

Spark's real argument is not speed on one machine. It is what happens when the data does **not** fit. pandas needs all the rows in memory at once; Spark processes them partition by partition, can spill to disk, and can spread the partitions over many machines. Combine the bytes per row of Section 1 with the measured cost per row, and estimate the limits of this machine:

In [ ]:
per_row_pd = np.polyfit(results["rows"], results["pandas_s"], 1)[0]
limit_rows = usable / slim_row
print(f"pandas, slim table: about {limit_rows / 1e6:,.0f} million rows fit; the question would then take "
      f"about {per_row_pd * limit_rows:,.0f} s.")
print(f"That is about {limit_rows / len(slim):,.0f} times today's AdventureWorks sales.")

**What you should see:** about 470 million slim rows fit (on a 16 GB machine), the question would then take about five and a half minutes in pandas, and that is almost 8,000 times today's AdventureWorks sales.

Beyond that point there are three choices, in order of cost and complexity:

1. **Make the data smaller**: fewer columns, compact types, Parquet, a summary table (Notebook 14).
2. **Process it in pieces on one machine**: stream through files, or use a single-machine engine that does it for you.
3. **Use a cluster**: Spark on many machines, or a cloud warehouse that hides the cluster (Redshift, Synapse, BigQuery). You gain capacity and pay in money, in start-up time, and in complexity.

For a data set the size of AdventureWorks, the honest recommendation is option 1 on one machine, and that is a perfectly good answer to give, provided it comes with the measurements that justify it and a statement of the growth at which the answer would change.

## ⚠️ Common pitfalls

* **Timing once.** One run can be disturbed by anything; take the best (or the median) of several.
* **Timing different things**: loading in one tool and only the query in another. Decide what is being measured, and measure the same thing everywhere.
* **Counting Spark's start-up against it**, or forgetting that a real job pays it too. Say which one you measured.
* **Checking speed but not answers.** Compare the results of every tool.
* **Generalising from one machine.** Report the cores and the memory with the times.

## 5. Where we stand

| Size of the data | Sensible tool | Why |
|---|---|---|
| Fits in memory many times over (like AdventureWorks) | pandas, or SQL in a single-file database | no overhead; simplest to run and to explain |
| Fits on one machine's disk, not in its memory | stream through files, Parquet, a single-machine engine | avoids a cluster's cost and complexity |
| Larger than one machine, or growing fast | Spark on a cluster, or a cloud warehouse | capacity, parallelism, fault tolerance |

Module E took one real warehouse through the whole journey: it was profiled (13), built with keys, views and indexes (14), reshaped into documents (15), explored with Spark (16), used to train and evaluate models (17), and finally measured (18). The tools changed at every step; the method did not. Measure the data, state the assumptions, choose the simplest tool that is enough, and know at what size that choice stops being right.

---
## ✏️ Exercises

## Level 1: warm-up

Reading and small calculations with the results of this notebook.

### Exercise 1 (Bytes per row of another table)
How many bytes per row does `DimCustomer` take in pandas? How many customers would fit in half of this machine's memory?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
cust = load("DimCustomer")
cust_row = cust.memory_usage(deep=True).sum() / len(cust)
print(f"DimCustomer: {cust_row:.0f} bytes per row; about {usable / cust_row / 1e6:,.0f} million customers would fit")
```

*About 1,390 bytes per customer, almost three times a sales row, because the customer table is full of text (names, addresses, e-mails). Still, about 6 million customers fit in half of a 16 GB machine: more than 300 times today's 18,484.*

</details>

### Exercise 2 (Who won at each size?)
From the `results` table, print for every size the fastest tool and how many times faster it was than the slowest.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
times = results.set_index("rows")[["pandas_s", "sqlite_s", "spark_s"]]
for rows, row in times.iterrows():
    print(f"{rows:>11,} rows: fastest {row.idxmin()[:-2]:<7} {row.max() / row.min():5.1f} times faster than {row.idxmax()[:-2]}")
```

*pandas is the fastest at every size. At the smallest size it is about 160 times faster than Spark; at the largest only about 6 times faster than SQLite, which is now the slowest. The slowest tool changes with size: Spark at the small end, SQLite at the large end.*

</details>

### Exercise 3 (Rows per second)
Turn the times into **throughput**: millions of rows per second for each tool at each size. At which size is each tool most efficient?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
throughput = results[["rows"]].copy()
for column in ["pandas_s", "sqlite_s", "spark_s"]:
    throughput[column[:-2]] = (results["rows"] / results[column] / 1e6).round(2)
throughput
```

*pandas handles about 1.4 to 2.1 million rows per second at every size, SQLite steadily about 0.2 to 0.3 million. Spark rises from about 0.01 to about 1.1 million rows per second as the data grows: its fixed cost is spread over more and more rows. A tool with a high overhead is most efficient on the largest data you give it.*

</details>

## Level 2: think it through

These questions are about **reasoning**. Decide first what you would need to see, then write the code that shows it.

### Exercise 4 (Is it the question or the overhead?)
Spark's times barely grow at small sizes. **How could you find out how much of Spark's time is fixed overhead, without any data at all?** Time a Spark query on a table with a single row.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
tiny = spark.createDataFrame(pd.DataFrame({"year": [2013], "product_key": [310], "customer_key": [1], "revenue": [1.0]}))
tiny_question = lambda: (tiny.join(F.broadcast(scat), "product_key").groupBy("year", "category")
                             .agg(F.sum("revenue"), F.countDistinct("customer_key")).collect())
tiny_question()
_, t_tiny = best_of(tiny_question)
print(f"Spark on ONE row: {t_tiny:.2f} s")
print("Spark on 1 copy: ", round(results.loc[0, "spark_s"], 2), "s")
```

*About 12 seconds for one row, even more than the 6.6 seconds for 60,398 rows. At this size Spark's time is almost all fixed cost, and how the tiny table was made (from pandas rather than from a Parquet file) changes that cost more than the rows do. Spark does not charge by the row on small data; it charges by the question.*

</details>

### Exercise 5 (A cheaper question)
**Predict first:** if the question drops the distinct count and keeps only `SUM(revenue)` by year and category, will the gap between pandas and Spark at 64 copies grow or shrink? Then measure.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
df64 = copies(64)
_, t_pd = best_of(lambda: df64.merge(category, on="product_key").groupby(["year", "category"])["revenue"].sum())
df64.to_parquet("sales_copies.parquet", index=False)
s64 = spark.read.parquet("sales_copies.parquet")
cheap = lambda: s64.join(F.broadcast(scat), "product_key").groupBy("year", "category").agg(F.sum("revenue")).collect()
cheap()
_, t_sp = best_of(cheap)
print(f"sum only, 64 copies:  pandas {t_pd:.2f} s   Spark {t_sp:.2f} s")
print(results.loc[results['copies'] == 64, ['pandas_s', 'spark_s']])
```

*The gap grows: pandas is about 8 times faster on the sum alone (0.84 s against 6.6 s), against about 4.6 times on the original question (1.85 s against 8.6 s). Spark's time hardly falls, because its fixed overhead is the same, while pandas saves the work of the distinct count. The cheaper the question, the larger the share of Spark's overhead in its time.*

</details>

### Exercise 6 (What would change the recommendation?)
Using `limit_rows` from Section 4, **how many years** could the business keep growing before one machine is no longer enough for pandas, if its sales grew by 50% a year? Write the calculation, then say which assumption in it you trust least.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
years, rows = 0, len(slim)
while rows < limit_rows:
    rows *= 1.5
    years += 1
print(f"At 50% growth a year, the slim table outgrows half of this machine's memory after about {years} years.")
```

*About 23 years at 50% growth a year, because the limit is almost 8,000 times today's sales. The weakest assumption is the growth rate itself, followed by the slim table: a question that needs more columns or text would reach the limit much sooner. Saying which assumption you trust least is part of a defensible answer.*

</details>

## Level 3: technical

Longer measurements.

### Exercise 7 (Parquet for pandas too)
Spark read Parquet; pandas worked on a table already in memory. Time how long pandas takes to **read** `sales_copies.parquet` for 64 copies, first all columns and then only `year` and `revenue`, and compare with reading the same rows from CSV.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
df64 = copies(64)
df64.to_parquet("sales_copies.parquet", index=False)
df64.to_csv("sales_copies.csv", index=False)
_, t_all = best_of(lambda: pd.read_parquet("sales_copies.parquet"), repeats=1)
_, t_two = best_of(lambda: pd.read_parquet("sales_copies.parquet", columns=["year", "revenue"]), repeats=1)
_, t_csv = best_of(lambda: pd.read_csv("sales_copies.csv"), repeats=1)
print(f"Parquet, all columns: {t_all:.2f} s   Parquet, two columns: {t_two:.2f} s   CSV: {t_csv:.2f} s")
print(f"sizes on disk: Parquet {os.path.getsize('sales_copies.parquet') / 1e6:.0f} MB, "
      f"CSV {os.path.getsize('sales_copies.csv') / 1e6:.0f} MB")
```

*Parquet is read in about 0.14 s (all columns) and 0.05 s (two columns), CSV in about 1.2 s: almost ten times slower. On disk, Parquet takes 20 MB and CSV 93 MB. Loading is often slower than the question itself, and the file format decides it; on a cloud service that charges per byte read, the same ratio becomes the bill.*

</details>

### Exercise 8 (Partitions)
Run the Spark question at 64 copies with the file read as 1, 8 and 32 partitions (`sdf.repartition(n)` before the question, cached and counted first). What happens, and why?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
df64 = copies(64)
df64.to_parquet("sales_copies.parquet", index=False)
for n in (1, 8, 32):
    part = spark.read.parquet("sales_copies.parquet").repartition(n).cache()
    part.count()
    _, t = best_of(lambda: ask_spark(part, scat))
    print(f"{n:>3} partitions: {t:.2f} s")
    part.unpersist()
```

*About 10.6, 10.4 and 9.4 seconds: almost no difference. On this data Spark's fixed overhead dominates, so how the work is cut into pieces hardly matters, which is this notebook's lesson again. On data large enough for the work to dominate, one partition would leave all but one core idle, one to a few partitions per core would use the machine fully, and many more would add scheduling cost for little gain.*

</details>

### Exercise 9 (A recommendation in numbers, a little harder)
Write a short function `recommend(rows, bytes_per_row, machine_gb)` that returns `"pandas on one machine"`, `"one machine, in pieces"` or `"a cluster"`, using: fits in half the memory, fits in ten times the memory (on disk), or more. Apply it to today's sales and to 1,000 and 100,000 times today's sales, on a 16 GB machine.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def recommend(rows, bytes_per_row, machine_gb):
    need_gb = rows * bytes_per_row / 1e9
    if need_gb <= machine_gb * 0.5:
        return "pandas on one machine"
    if need_gb <= machine_gb * 10:
        return "one machine, in pieces"
    return "a cluster"

for growth in (1, 1_000, 100_000):
    rows = len(slim) * growth
    print(f"{growth:>7}x today ({rows:>14,} rows): {recommend(rows, slim_row, 16)}")
```

*Today's sales and even 1,000 times today's fit comfortably in pandas on one machine; 100,000 times today's (about 6 billion rows, about 109 GB) would need processing in pieces, but still not a cluster. Crude, but explicit: anyone can check the assumptions and the arithmetic, and change them. That is what makes a recommendation defensible, which a sentence such as 'Spark is better for big data' is not.*

</details>

## 🔑 Key takeaways

- Measure the **bytes per row** first; choosing columns and types often matters more than choosing an engine.
- Time **only the question**, warm up first, repeat, and check that every tool returns the **same answer**.
- A tool's time is a **fixed overhead** plus a **cost per row**; on small data the overhead decides.
- On data that fits in memory, pandas usually wins; Spark's case is data that does **not** fit, and capacity across machines.
- A defensible recommendation states the **measurements**, the **assumptions**, and the **size at which it would change**.

### 🎓 Congratulations: you have completed Module E, and the whole course!

You took one real data warehouse through every idea of the course: you profiled it, built it with the rules that keep it true, reshaped it into documents, explored it and trained models on it with Spark, and finally measured what scale would mean for it. On the way the measurements kept their habit of surprising: the simplest tool often won, and the reasons why it won were more instructive than the result.

The questions to carry with you: How big is the data, really? How is it read, and how does it change? What must never be wrong? And what, exactly, am I giving up to get this?

In [ ]:
spark.stop()

---
*End of Module E: and the end of the course.*

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*